In [1]:
# ════════════════════════════════════════════════════════════════════════
# ABLATION CELL 1 — CONFIG + IMPORTS + SEED   (run once)
# ════════════════════════════════════════════════════════════════════════
import os, gc, json, time, random, warnings
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.amp import GradScaler, autocast
import timm, cv2
from PIL import Image
import torchvision.transforms as transforms
from sklearn.metrics import precision_recall_fscore_support

warnings.filterwarnings('ignore')

# ── EDIT THESE PATHS ──
CSV_PATH   = '/home/sufi/merged_dataset_metadata_augmented.csv'
OUTPUT_DIR = Path('/home/sufi/training_results/ablation')
RESULTS_DIR = OUTPUT_DIR / 'run_results'   # per-run JSONs land here
CKPT_DIR    = OUTPUT_DIR / 'checkpoints'   # per-run weights land here
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# ── Protocol (locked) ──
SEED          = 42
EPOCHS        = 50
BATCH_SIZE    = 32
NUM_WORKERS   = 4
IMG_SIZE      = 224
GRAD_CLIP     = 1.0
WEIGHT_DECAY  = 1e-4
EMA_DECAY     = 0.9995
FOCAL_GAMMA   = 2.0
LR_BACKBONE   = 1e-4
LR_CA         = 5e-4
LR_HEAD       = 1e-3
FREEZE_EPOCHS = 3
EMA_START     = 5

SEMANTIC_GROUPS = ['contamination','cut','deformation','fracture',
                   'hole_void','minor_defect','scratch','surface_quality']
NUM_DEFECT_TYPES  = len(SEMANTIC_GROUPS)
NUM_PRODUCT_TYPES = 17
DEFECT_TYPE_TO_IDX = {n: i for i, n in enumerate(SEMANTIC_GROUPS)}

DEFECT_GROUP_MAP = {
    'crack':'fracture','fracture':'fracture','faulty_imprint':'fracture',
    'hole':'hole_void','void':'hole_void','pit':'hole_void','blowhole':'hole_void',
    'scratch':'scratch','score':'scratch',
    'stain':'surface_quality','color':'surface_quality','rough':'surface_quality',
    'uneven':'surface_quality','inclusion':'surface_quality',
    'discolor':'surface_quality','pilling':'surface_quality',
    'bent':'deformation','bent_lead':'deformation','squeeze':'deformation',
    'deformation':'deformation',
    'contamination':'contamination','glue':'contamination','oil':'contamination',
    'glue_strip':'contamination','liquid':'contamination',
    'metal_contamination':'contamination',
    'missing':'minor_defect','misplaced':'minor_defect','flip':'minor_defect',
    'missing_hole':'minor_defect','thread':'minor_defect','cable_swap':'minor_defect',
    'combined':'minor_defect','cut':'cut',
    'hole_void':'hole_void','surface_quality':'surface_quality',
    'minor_defect':'minor_defect',
}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def _seed_worker(worker_id):
    np.random.seed(SEED + worker_id); random.seed(SEED + worker_id)

print(f"✅ CELL 1 done — device={device}, output={OUTPUT_DIR}")

/home/sufi/miniconda3/envs/organized/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ CELL 1 done — device=cuda, output=/home/sufi/training_results/ablation


In [2]:
# ════════════════════════════════════════════════════════════════════════
# ABLATION CELL 2 — DATA   (run once)
# ════════════════════════════════════════════════════════════════════════
def load_dataframes():
    df = pd.read_csv(CSV_PATH)
    if 'path' in df.columns and 'image_path' not in df.columns:
        df = df.rename(columns={'path': 'image_path'})
    if 'category' in df.columns and 'product_type' not in df.columns:
        df = df.rename(columns={'category': 'product_type'})

    def cb(v):
        if pd.isna(v): return 0
        return 0 if str(v).strip().lower() in ('0','good','normal','ok','casting_ok') else 1
    df['binary_label'] = df['label'].apply(cb)

    def rm(raw):
        if pd.isna(raw): return -1
        s = str(raw).strip().lower()
        if s in ('good','normal','casting_ok',''): return -1
        sem = DEFECT_GROUP_MAP.get(s, None)
        if sem is None and s in SEMANTIC_GROUPS: sem = s
        if sem is None:
            for k, v in DEFECT_GROUP_MAP.items():
                if k in s: sem = v; break
        return -1 if sem is None else DEFECT_TYPE_TO_IDX.get(sem, -1)
    df['defect_type_label'] = df['defect_type'].apply(rm)
    df.loc[df['binary_label'] == 0, 'defect_type_label'] = -1

    prods = sorted(df['product_type'].dropna().unique().tolist())
    p2i = {p: i for i, p in enumerate(prods)}
    df['product_type_label'] = [p2i.get(str(v), 0) if not pd.isna(v) else 0
                                for v in df['product_type']]

    tr = df[df['split']=='train'].reset_index(drop=True)
    va = df[df['split']=='val'].reset_index(drop=True)
    te = df[df['split']=='test'].reset_index(drop=True)
    return tr, va, te

class ThreeHeadDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True); self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(row['image_path'])
        img = (np.zeros((IMG_SIZE,IMG_SIZE,3),np.uint8) if img is None
               else cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        img = Image.fromarray(img)
        if self.transform: img = self.transform(img)
        return (img, int(row['binary_label']), int(row['defect_type_label']),
                int(row['product_type_label']), row['image_path'])

TRAIN_TF = transforms.Compose([
    transforms.Resize((IMG_SIZE,IMG_SIZE)),
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15), transforms.ColorJitter(0.3,0.3,0.2,0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
EVAL_TF = transforms.Compose([
    transforms.Resize((IMG_SIZE,IMG_SIZE)), transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

def make_weighted_sampler(df):
    counts = np.ones(NUM_DEFECT_TYPES)
    for i in range(NUM_DEFECT_TYPES):
        counts[i] = max(1, (df['defect_type_label']==i).sum())
    cw = 1.0/counts; cw = cw/cw.sum()
    w = [0.3 if int(r)==-1 else float(cw[int(r)]) for r in df['defect_type_label']]
    return WeightedRandomSampler(torch.tensor(w,dtype=torch.float32), len(w), replacement=True)

def cutmix_collate(batch):
    imgs, bl, dl, pl, paths = zip(*batch)
    imgs = torch.stack(imgs)
    bl = torch.tensor(bl,dtype=torch.long); dl = torch.tensor(dl,dtype=torch.long)
    pl = torch.tensor(pl,dtype=torch.long)
    known = (dl != -1).nonzero(as_tuple=True)[0]
    if len(known) >= 4 and torch.rand(1).item() < 0.5:
        lam = max(0.3, min(0.7, float(np.random.beta(1.0,1.0))))
        perm = known[torch.randperm(len(known))]
        _,_,H,W = imgs.shape
        cr = np.sqrt(1-lam); ch_,cw_ = int(H*cr), int(W*cr)
        cx,cy = np.random.randint(W), np.random.randint(H)
        x1,x2 = max(0,cx-cw_//2), min(W,cx+cw_//2)
        y1,y2 = max(0,cy-ch_//2), min(H,cy+ch_//2)
        ic = imgs.clone(); ic[known,:,y1:y2,x1:x2] = imgs[perm,:,y1:y2,x1:x2]
        lam_a = 1-(x2-x1)*(y2-y1)/(H*W); imgs = ic
        if lam_a < 0.5:
            dn = dl.clone(); dn[known] = dl[perm]; dl = dn
    return imgs, bl, dl, pl, list(paths)

def build_loaders(tr, va, te, use_cutmix):
    g = torch.Generator(); g.manual_seed(SEED)
    tl = DataLoader(ThreeHeadDataset(tr,TRAIN_TF), BATCH_SIZE,
                    sampler=make_weighted_sampler(tr), num_workers=NUM_WORKERS,
                    pin_memory=True, drop_last=True,
                    collate_fn=(cutmix_collate if use_cutmix else None),
                    worker_init_fn=_seed_worker, generator=g)
    vl = DataLoader(ThreeHeadDataset(va,EVAL_TF), BATCH_SIZE, shuffle=False,
                    num_workers=NUM_WORKERS, pin_memory=True,
                    worker_init_fn=_seed_worker, generator=g)
    tel = DataLoader(ThreeHeadDataset(te,EVAL_TF), BATCH_SIZE, shuffle=False,
                     num_workers=NUM_WORKERS, pin_memory=True,
                     worker_init_fn=_seed_worker, generator=g)
    return tl, vl, tel

# Load once and cache in memory
train_df, val_df, test_df = load_dataframes()
print(f"✅ CELL 2 done — train={len(train_df):,} val={len(val_df):,} test={len(test_df):,}")

✅ CELL 2 done — train=16,337 val=3,339 test=1,668


In [3]:
# ════════════════════════════════════════════════════════════════════════
# ABLATION CELL 3 — MODEL (feature-flagged)   (run once)
# ════════════════════════════════════════════════════════════════════════
class CoordAttention(nn.Module):
    def __init__(self, ch, reduction=32):
        super().__init__()
        mid = max(8, ch//reduction)
        self.conv1 = nn.Conv2d(ch, mid, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(mid); self.act = nn.ReLU(inplace=True)
        self.conv_h = nn.Conv2d(mid, ch, 1, bias=True)
        self.conv_w = nn.Conv2d(mid, ch, 1, bias=True)
        self._id_init()
    def _id_init(self):
        nn.init.zeros_(self.conv_h.weight); nn.init.constant_(self.conv_h.bias, 10.0)
        nn.init.zeros_(self.conv_w.weight); nn.init.constant_(self.conv_w.bias, 10.0)
    def forward(self, x):
        B,C,H,W = x.shape
        x_h = x.mean(3, keepdim=True)
        x_w = x.mean(2, keepdim=True).permute(0,1,3,2)
        y = self.act(self.bn1(self.conv1(torch.cat([x_h,x_w], dim=2))))
        x_h, x_w = torch.split(y, [H,W], dim=2)
        x_w = x_w.permute(0,1,3,2)
        return x * torch.sigmoid(self.conv_h(x_h)) * torch.sigmoid(self.conv_w(x_w))

class AblationModel(nn.Module):
    def __init__(self, use_b2=True, use_coord_attn=True, use_multiscale=True,
                 num_defect=NUM_DEFECT_TYPES, num_product=NUM_PRODUCT_TYPES):
        super().__init__()
        self.use_coord_attn = use_coord_attn
        self.use_multiscale = use_multiscale
        name = 'efficientnet_b2' if use_b2 else 'efficientnet_b0'
        self.backbone = timm.create_model(name, pretrained=True,
                                          features_only=True, out_indices=(2,3,4))
        c2,c3,c4 = self.backbone.feature_info.channels()
        self.ca_mid  = CoordAttention(c3) if use_coord_attn else nn.Identity()
        self.ca_deep = CoordAttention(c4) if use_coord_attn else nn.Identity()
        self.neck = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(c4,512,bias=False), nn.BatchNorm1d(512), nn.SiLU(), nn.Dropout(0.35))
        self.binary_head = nn.Linear(512, 1)
        self.product_head = nn.Sequential(
            nn.Linear(512,256,bias=False), nn.BatchNorm1d(256), nn.SiLU(),
            nn.Dropout(0.2), nn.Linear(256, num_product))
        self.defect_gap = nn.AdaptiveAvgPool2d(1)
        din = (c2+c3+c4) if use_multiscale else c4
        self.defect_head = nn.Sequential(
            nn.Linear(din,384,bias=False), nn.BatchNorm1d(384), nn.SiLU(), nn.Dropout(0.3),
            nn.Linear(384,192,bias=False), nn.BatchNorm1d(192), nn.SiLU(), nn.Dropout(0.2),
            nn.Linear(192, num_defect))
        self._init_heads()
        if use_coord_attn:
            self.ca_mid._id_init(); self.ca_deep._id_init()
    def _init_heads(self):
        for m in [self.neck, self.binary_head, self.product_head, self.defect_head]:
            for l in m.modules():
                if isinstance(l, nn.Linear):
                    nn.init.xavier_uniform_(l.weight)
                    if l.bias is not None: nn.init.zeros_(l.bias)
                elif isinstance(l, (nn.BatchNorm1d, nn.BatchNorm2d)):
                    nn.init.ones_(l.weight); nn.init.zeros_(l.bias)
    def forward(self, x):
        f2,f3,f4 = self.backbone(x)
        f3 = self.ca_mid(f3); f4 = self.ca_deep(f4)
        shared = self.neck(f4)
        bo = self.binary_head(shared); po = self.product_head(shared)
        if self.use_multiscale:
            d2 = self.defect_gap(f2).flatten(1); d3 = self.defect_gap(f3).flatten(1)
            d4 = self.defect_gap(f4).flatten(1)
            df = torch.cat([d2,d3,d4], dim=1)
        else:
            df = self.defect_gap(f4).flatten(1)
        return bo, self.defect_head(df), po
    def count_params(self):
        return sum(p.numel() for p in self.parameters())/1e6

print("✅ CELL 3 done — AblationModel ready")

✅ CELL 3 done — AblationModel ready


In [4]:
# ════════════════════════════════════════════════════════════════════════
# ABLATION CELL 4 — LOSSES + EMA + METRICS   (run once)
# ════════════════════════════════════════════════════════════════════════
class PlainDefectLoss(nn.Module):
    def __init__(self, ignore_index=-1): super().__init__(); self.ig=ignore_index
    def forward(self, logits, t):
        v = t != self.ig
        if v.sum()==0: return logits.sum()*0.0
        return F.cross_entropy(logits[v], t[v])

class WeightedFocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None, ignore_index=-1):
        super().__init__(); self.g=gamma; self.w=weight; self.ig=ignore_index
    def forward(self, logits, t):
        v = t != self.ig
        if v.sum()==0: return logits.sum()*0.0
        tt, ll = t[v], logits[v]
        ce = F.cross_entropy(ll, tt, reduction='none')
        pt = torch.exp(-ce); foc = (1-pt)**self.g
        if self.w is not None: return (foc*self.w[tt]*ce).mean()
        return (foc*ce).mean()

class UncertaintyWeighting(nn.Module):
    def __init__(self, enabled=True, warmup=10):
        super().__init__(); self.enabled=enabled; self.warmup=warmup; self.epoch=0
        self.log_vars = nn.Parameter(torch.zeros(3))
        self.register_buffer('fixed_w', torch.tensor([0.35,0.40,0.25]))
    def set_epoch(self, e): self.epoch=e
    def forward(self, lb, ld, lp):
        if (not self.enabled) or self.epoch <= self.warmup:
            w=self.fixed_w; return w[0]*lb + w[1]*ld + w[2]*lp
        lv = self.log_vars.clamp(-3,3); p = torch.exp(-lv).clamp(0.05,15)
        tot = p[0]*lb+lv[0] + p[1]*ld+lv[1] + p[2]*lp+lv[2]
        if tot.item() < 0:
            w=self.fixed_w; return w[0]*lb + w[1]*ld + w[2]*lp
        return tot

def compute_class_weights(loader):
    c = torch.zeros(NUM_DEFECT_TYPES)
    with torch.no_grad():
        for _,_,dl,_,_ in loader:
            v = dl[dl!=-1]
            for i in range(NUM_DEFECT_TYPES): c[i] += (v==i).sum()
    w = c.sum()/(c.clamp(min=1)*NUM_DEFECT_TYPES)
    return (w/w.mean()).to(device)

class EMA:
    def __init__(self, model, decay=EMA_DECAY):
        self.decay=decay
        self.shadow={k:v.clone().detach() for k,v in model.state_dict().items()}
    def reseed(self, model):
        self.shadow={k:v.clone().detach() for k,v in model.state_dict().items()}
    def update(self, model):
        for k,v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1-self.decay)
    def apply(self, model):
        self._b={k:v.clone() for k,v in model.state_dict().items()}
        for k,v in model.state_dict().items(): v.copy_(self.shadow[k])
    def restore(self, model):
        for k,v in model.state_dict().items(): v.copy_(self._b[k])
        del self._b

def defect_metrics(preds, labels):
    m = labels != -1
    if m.sum()==0: return 0.0,0.0,np.zeros(NUM_DEFECT_TYPES)
    p,r,f,_ = precision_recall_fscore_support(labels[m], preds[m], average=None,
        labels=list(range(NUM_DEFECT_TYPES)), zero_division=0)
    return float(f.mean()), float((preds[m]==labels[m]).mean()*100), f

@torch.no_grad()
def evaluate(model, loader, ema=None):
    if ema is not None: ema.apply(model)
    model.eval(); tot=bc=pc=0; ap=[]; at=[]
    for imgs,bl,dl,pl,_ in loader:
        imgs=imgs.to(device); bl=bl.to(device); pl=pl.to(device)
        with autocast('cuda', enabled=torch.cuda.is_available()):
            sb,sd,sp = model(imgs); sb=sb.squeeze(1)
        tot+=imgs.size(0)
        bc += ((torch.sigmoid(sb)>0.5).long()==bl).sum().item()
        pc += (sp.argmax(1)==pl).sum().item()
        ap.extend(sd.argmax(1).cpu().numpy()); at.extend(dl.numpy())
    if ema is not None: ema.restore(model)
    f1,da,pf = defect_metrics(np.array(ap), np.array(at))
    return {'macro_f1':f1,'defect_acc':da,'binary_acc':100*bc/tot,
            'product_acc':100*pc/tot,'per_class_f1':pf}

@torch.no_grad()
def measure_latency(model, n_warm=30, n_meas=200):
    model.eval(); x=torch.randn(1,3,IMG_SIZE,IMG_SIZE,device=device)
    for _ in range(n_warm): _=model(x)
    if torch.cuda.is_available(): torch.cuda.synchronize()
    t0=time.time()
    for _ in range(n_meas): _=model(x)
    if torch.cuda.is_available(): torch.cuda.synchronize()
    ms=(time.time()-t0)/n_meas*1000
    return ms, 1000/ms

print("✅ CELL 4 done — losses, EMA, metrics ready")

✅ CELL 4 done — losses, EMA, metrics ready


In [5]:
# ════════════════════════════════════════════════════════════════════════
# ABLATION CELL 5 — SINGLE-RUN TRAINER   (run once to define)
# ════════════════════════════════════════════════════════════════════════
def train_one_config(cfg, verbose=True):
    """Train ONE config, save result JSON + checkpoint immediately, return result."""
    name = cfg['name']
    safe = name.replace(' ','_').replace('+','').replace('/','-').replace('.','')
    result_path = RESULTS_DIR / f"{safe}.json"
    ckpt_path   = CKPT_DIR / f"{safe}.pth"

    # ── SKIP if already done (resume capability) ──
    if result_path.exists():
        print(f"⏭  SKIP '{name}' — already done ({result_path.name})")
        with open(result_path) as f: return json.load(f)

    print(f"\n{'='*70}\n RUN: {name}\n{'='*70}")
    print(f"  b2={cfg['use_b2']} coord={cfg['use_coord_attn']} ms={cfg['use_multiscale']} "
          f"wf={cfg['use_weighted_focal']} warm={cfg['use_warm_restart']} "
          f"cutmix={cfg['use_cutmix']} uw={cfg['use_uncertainty']} ema={cfg['use_ema']}")

    set_seed(SEED)
    gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
    tl, vl, tel = build_loaders(train_df, val_df, test_df, use_cutmix=cfg['use_cutmix'])

    model = AblationModel(cfg['use_b2'], cfg['use_coord_attn'],
                          cfg['use_multiscale']).to(device)
    n_params = model.count_params()

    if cfg['use_weighted_focal']:
        defect_loss = WeightedFocalLoss(FOCAL_GAMMA, compute_class_weights(tl)).to(device)
    else:
        defect_loss = PlainDefectLoss().to(device)
    bin_loss = nn.BCEWithLogitsLoss().to(device)
    prod_loss = nn.CrossEntropyLoss().to(device)
    uw = UncertaintyWeighting(cfg['use_uncertainty']).to(device)

    ca_p = (list(model.ca_mid.parameters())+list(model.ca_deep.parameters())) \
           if cfg['use_coord_attn'] else []
    head_p = (list(model.neck.parameters())+list(model.binary_head.parameters())+
              list(model.product_head.parameters())+list(model.defect_head.parameters())+
              list(uw.parameters()))
    opt = torch.optim.AdamW([
        {'params':list(model.backbone.parameters()),'lr':LR_BACKBONE},
        {'params':ca_p,'lr':LR_CA},
        {'params':head_p,'lr':LR_HEAD}], weight_decay=WEIGHT_DECAY)
    for p in model.backbone.parameters(): p.requires_grad=False
    unfrozen=False

    if cfg['use_warm_restart']:
        sched = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=30, T_mult=2, eta_min=1e-6)
    else:
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-6)

    scaler = GradScaler('cuda', enabled=torch.cuda.is_available())
    ema = EMA(model) if cfg['use_ema'] else None
    best_f1=0.0; best_state=None; t0=time.time()

    for epoch in range(1, EPOCHS+1):
        if epoch==FREEZE_EPOCHS+1 and not unfrozen:
            for p in model.backbone.parameters(): p.requires_grad=True
            unfrozen=True
        if ema is not None and epoch==EMA_START: ema.reseed(model)
        uw.set_epoch(epoch)

        model.train()
        for imgs,bl,dl,pl,_ in tl:
            imgs=imgs.to(device); bl=bl.to(device); dl=dl.to(device); pl=pl.to(device)
            opt.zero_grad()
            with autocast('cuda', enabled=torch.cuda.is_available()):
                sb,sd,sp = model(imgs); sb=sb.squeeze(1)
                loss = uw(bin_loss(sb,bl.float()), defect_loss(sd,dl), prod_loss(sp,pl))
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(opt); scaler.update()
            if ema is not None: ema.update(model)
        sched.step()

        use_ema = ema is not None and epoch>=EMA_START
        vm = evaluate(model, vl, ema=ema if use_ema else None)
        if vm['macro_f1'] > best_f1:
            best_f1 = vm['macro_f1']
            if use_ema: ema.apply(model)
            best_state = {k:v.clone() for k,v in model.state_dict().items()}
            if use_ema: ema.restore(model)

        if verbose and (epoch%10==0 or epoch==1 or epoch==EPOCHS):
            print(f"  ep {epoch:>3}/{EPOCHS}  val_F1={vm['macro_f1']:.4f}  "
                  f"def={vm['defect_acc']:.1f}% bin={vm['binary_acc']:.1f}% "
                  f"prod={vm['product_acc']:.1f}%  best={best_f1:.4f}")

    if best_state is not None: model.load_state_dict(best_state)
    tm = evaluate(model, tel, ema=None)
    ms, fps = measure_latency(model)

    torch.save({'name':name,'cfg':cfg,'model_state_dict':model.state_dict(),
                'val_best_f1':best_f1,'params_M':n_params}, ckpt_path)

    result = {'name':name,'macro_f1':tm['macro_f1'],'defect_acc':tm['defect_acc'],
              'binary_acc':tm['binary_acc'],'product_acc':tm['product_acc'],
              'params_M':round(n_params,2),'latency_ms':round(ms,2),'fps':round(fps,1),
              'per_class_f1':tm['per_class_f1'].tolist(),'val_best_f1':best_f1,
              'minutes':round((time.time()-t0)/60,1)}
    with open(result_path,'w') as f: json.dump(result, f, indent=2)

    print(f"  → TEST F1={tm['macro_f1']:.4f} def={tm['defect_acc']:.1f}% "
          f"bin={tm['binary_acc']:.1f}% prod={tm['product_acc']:.1f}% "
          f"{round(n_params,2)}M {round(ms,2)}ms/{round(fps,1)}fps "
          f"({result['minutes']}min)")
    print(f"  💾 saved {result_path.name}")

    del model, opt, sched, scaler, ema
    gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
    return result

print("✅ CELL 5 done — train_one_config() ready")

✅ CELL 5 done — train_one_config() ready


In [6]:
# ════════════════════════════════════════════════════════════════════════
# ABLATION CELL 6 — THE 7 CONFIGS   (run once)
# ════════════════════════════════════════════════════════════════════════
def _cfg(name,b2,coord,ms,wf,warm,cutmix,uw,ema):
    return dict(name=name, use_b2=b2, use_coord_attn=coord, use_multiscale=ms,
                use_weighted_focal=wf, use_warm_restart=warm,
                use_cutmix=cutmix, use_uncertainty=uw, use_ema=ema)

ABLATION_CONFIGS = [
    #     name                            b2     coord  ms     wf     warm   cutmix uw     ema
    _cfg("1. Baseline (B0, MTL)",         False, False, False, False, False, False, False, False),
    _cfg("2. + B2 backbone",              True,  False, False, False, False, False, False, False),
    _cfg("3. + CoordAttention",           True,  True,  False, False, False, False, False, False),
    _cfg("4. + Multi-scale head",         True,  True,  True,  False, False, False, False, False),
    _cfg("5. + Weighted focal loss",      True,  True,  True,  True,  False, False, False, False),
    _cfg("6. + Warm restart",             True,  True,  True,  True,  True,  False, False, False),
    _cfg("7. + Training group (Full)",    True,  False,  True,  True,  True,  True,  True,  True),
]
for i,c in enumerate(ABLATION_CONFIGS): print(f"  [{i}] {c['name']}")
print(f"✅ CELL 6 done — {len(ABLATION_CONFIGS)} configs defined")

  [0] 1. Baseline (B0, MTL)
  [1] 2. + B2 backbone
  [2] 3. + CoordAttention
  [3] 4. + Multi-scale head
  [4] 5. + Weighted focal loss
  [5] 6. + Warm restart
  [6] 7. + Training group (Full)
✅ CELL 6 done — 7 configs defined


In [10]:
# ════════════════════════════════════════════════════════════════════════
# ABLATION CELL 7 — RUN ONE CONFIG AT A TIME
# ════════════════════════════════════════════════════════════════════════
# Change RUN_INDEX from 0 → 6, running this cell once per index.
# Each completed run is saved to disk, so you can stop/restart anytime.
# Already-completed runs auto-skip (safe to re-run).
#
# To run ALL in one go instead, see the loop at the bottom (commented out).
# ════════════════════════════════════════════════════════════════════════

RUN_INDEX = 6     # ← change this: 0,1,2,3,4,5,6

result = train_one_config(ABLATION_CONFIGS[RUN_INDEX])
print(f"\n✅ Run {RUN_INDEX} complete. Set RUN_INDEX={RUN_INDEX+1} and re-run this cell.")

# ── OR: uncomment to run all 7 back-to-back (auto-skips done ones) ──
# for c in ABLATION_CONFIGS:
#     train_one_config(c)
# print("✅ All runs complete")


 RUN: 7. + Training group (Full)
  b2=True coord=False ms=True wf=True warm=True cutmix=True uw=True ema=True


Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


  ep   1/50  val_F1=0.4750  def=52.7% bin=88.9% prod=99.6%  best=0.4750
  ep  10/50  val_F1=0.6511  def=74.0% bin=96.0% prod=99.9%  best=0.6511
  ep  20/50  val_F1=0.8492  def=87.8% bin=97.7% prod=99.9%  best=0.8492
  ep  30/50  val_F1=0.8852  def=90.2% bin=98.5% prod=99.9%  best=0.8886
  ep  40/50  val_F1=0.9311  def=93.9% bin=98.7% prod=100.0%  best=0.9311
  ep  50/50  val_F1=0.9401  def=94.6% bin=99.1% prod=100.0%  best=0.9446
  → TEST F1=0.9182 def=93.2% bin=98.4% prod=99.9% 7.8M 6.87ms/145.7fps (54.1min)
  💾 saved 7__Training_group_(Full).json

✅ Run 6 complete. Set RUN_INDEX=7 and re-run this cell.


In [13]:
# ════════════════════════════════════════════════════════════════════════
# ABLATION CELL 7 — RUN ONE CONFIG AT A TIME
# ════════════════════════════════════════════════════════════════════════
# Change RUN_INDEX from 0 → 6, running this cell once per index.
# Each completed run is saved to disk, so you can stop/restart anytime.
# Already-completed runs auto-skip (safe to re-run).
#
# To run ALL in one go instead, see the loop at the bottom (commented out).
# ════════════════════════════════════════════════════════════════════════

RUN_INDEX = 4     # ← change this: 0,1,2,3,4,5,6

result = train_one_config(ABLATION_CONFIGS[RUN_INDEX])
print(f"\n✅ Run {RUN_INDEX} complete. Set RUN_INDEX={RUN_INDEX+1} and re-run this cell.")

# ── OR: uncomment to run all 7 back-to-back (auto-skips done ones) ──
# for c in ABLATION_CONFIGS:
#     train_one_config(c)
# print("✅ All runs complete")

⏭  SKIP '5. + Weighted focal loss' — already done (5__Weighted_focal_loss.json)

✅ Run 4 complete. Set RUN_INDEX=5 and re-run this cell.


In [14]:
# ════════════════════════════════════════════════════════════════════════
# ABLATION CELL 8 — BUILD TABLES + MULTI-PANEL DASHBOARD   (run anytime)
# ════════════════════════════════════════════════════════════════════════
# Reads completed run JSONs from RESULTS_DIR. Works with partial results.
# Produces: ablation_results.csv, ablation_per_class_f1.csv,
#           ablation_dashboard.png (6 panels), ablation_staircase.png
# ════════════════════════════════════════════════════════════════════════
import json
import numpy as np
import pandas as pd

# ── Load results in config order ──
loaded = []
for c in ABLATION_CONFIGS:
    safe = c['name'].replace(' ','_').replace('+','').replace('/','-').replace('.','')
    rp = RESULTS_DIR / f"{safe}.json"
    if rp.exists():
        with open(rp) as f: loaded.append(json.load(f))
    else:
        print(f"⚠️  missing: {c['name']} (not run yet)")
print(f"\nFound {len(loaded)}/{len(ABLATION_CONFIGS)} completed runs\n")

if len(loaded) == 0:
    raise SystemExit("No completed runs yet — run Cell 7 first.")

# ── Short labels for x-axis (the leading number) ──
short = []
for r in loaded:
    n = r['name']
    short.append(n.split('.')[0].strip() if '.' in n[:3] else n[:12])

# ── Main table ──
rows=[]; prev=None
for r in loaded:
    delta = '' if prev is None else f"{r['macro_f1']-prev:+.4f}"
    rows.append({'Configuration':r['name'],'Macro_F1':round(r['macro_f1'],4),
                 'Delta_F1':delta,'Defect_Acc':round(r['defect_acc'],2),
                 'Binary_Acc':round(r['binary_acc'],2),'Product_Acc':round(r['product_acc'],2),
                 'Params_M':r['params_M'],'Latency_ms':r['latency_ms'],'FPS':r['fps']})
    prev = r['macro_f1']
main_df = pd.DataFrame(rows)
main_df.to_csv(OUTPUT_DIR/'ablation_results.csv', index=False)

# ── Per-class table ──
pc=[]
for r in loaded:
    row={'Configuration':r['name']}
    for i,cls in enumerate(SEMANTIC_GROUPS): row[cls]=round(float(r['per_class_f1'][i]),3)
    pc.append(row)
pc_df = pd.DataFrame(pc)
pc_df.to_csv(OUTPUT_DIR/'ablation_per_class_f1.csv', index=False)

# ════════════════════════════════════════════════════════════════════════
#  6-PANEL DASHBOARD
# ════════════════════════════════════════════════════════════════════════
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import cm

plt.rcParams.update({'font.size': 9, 'axes.grid': True, 'grid.alpha': 0.25})

f1s   = [r['macro_f1']    for r in loaded]
dacc  = [r['defect_acc']  for r in loaded]
bacc  = [r['binary_acc']  for r in loaded]
pacc  = [r['product_acc'] for r in loaded]
prm   = [r['params_M']    for r in loaded]
lat   = [r['latency_ms']  for r in loaded]
fps   = [r['fps']         for r in loaded]
x     = list(range(len(loaded)))

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle('EdgeNetV4 Ablation — Cumulative Component Contribution',
             fontsize=15, fontweight='bold', y=0.995)

# ── Panel 1: Three-head accuracy comparison (grouped bars) ──
ax = axes[0,0]
w = 0.26
ax.bar([i-w for i in x], bacc, w, label='Binary Acc',  color='#4C72B0')
ax.bar(x,               dacc, w, label='Defect Acc',  color='#DD8452')
ax.bar([i+w for i in x], pacc, w, label='Product Acc', color='#55A868')
ax.set_title('Three-Head Accuracy by Configuration', fontweight='bold')
ax.set_ylabel('Accuracy (%)')
ax.set_xticks(x); ax.set_xticklabels(short)
ax.set_ylim(min(min(dacc),min(bacc),min(pacc))-5, 101)
ax.legend(fontsize=8, loc='lower right')

# ── Panel 2: Macro-F1 staircase with deltas ──
ax = axes[0,1]
ax.plot(x, f1s, 'o-', lw=2.2, color='#2A9D8F', markersize=9)
for i,v in enumerate(f1s):
    ax.text(i, v+0.004, f"{v:.3f}", ha='center', fontsize=8, fontweight='bold')
    if i>0:
        d = f1s[i]-f1s[i-1]
        col = '#2E7D32' if d>=0 else '#C62828'
        ax.text(i-0.5, (f1s[i]+f1s[i-1])/2, f"{d:+.3f}",
                ha='center', fontsize=7.5, color=col, style='italic')
ax.set_title('Defect Macro-F1 Build-up (Δ vs previous)', fontweight='bold')
ax.set_ylabel('Test Macro F1')
ax.set_xticks(x); ax.set_xticklabels(short)

# ── Panel 3: Defect Acc vs Macro-F1 (dual axis) ──
ax = axes[0,2]
ax.plot(x, dacc, 's-', color='#DD8452', label='Defect Acc (%)', lw=2)
ax.set_ylabel('Defect Acc (%)', color='#DD8452')
ax.tick_params(axis='y', labelcolor='#DD8452')
ax2 = ax.twinx()
ax2.plot(x, f1s, 'o-', color='#2A9D8F', label='Macro F1', lw=2)
ax2.set_ylabel('Macro F1', color='#2A9D8F')
ax2.tick_params(axis='y', labelcolor='#2A9D8F')
ax2.grid(False)
ax.set_title('Defect Accuracy vs Macro-F1', fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(short)

# ── Panel 4: Per-class F1 heatmap ──
ax = axes[1,0]
mat = np.array([[float(r['per_class_f1'][i]) for i in range(NUM_DEFECT_TYPES)]
                for r in loaded])
im = ax.imshow(mat, aspect='auto', cmap='YlGnBu', vmin=max(0, mat.min()-0.05), vmax=1.0)
ax.set_xticks(range(NUM_DEFECT_TYPES))
ax.set_xticklabels(SEMANTIC_GROUPS, rotation=45, ha='right', fontsize=7.5)
ax.set_yticks(x); ax.set_yticklabels(short)
ax.set_title('Per-Class F1 Across Configs', fontweight='bold')
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        ax.text(j, i, f"{mat[i,j]:.2f}", ha='center', va='center',
                fontsize=6.5, color='black' if mat[i,j]<0.7 else 'white')
fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02)

# ── Panel 5: Params & Latency (cost) ──
ax = axes[1,1]
w = 0.38
ax.bar([i-w/2 for i in x], prm, w, label='Params (M)', color='#8172B3')
ax.set_ylabel('Params (M)', color='#8172B3')
ax.tick_params(axis='y', labelcolor='#8172B3')
ax3 = ax.twinx()
ax3.bar([i+w/2 for i in x], lat, w, label='Latency (ms)', color='#CCB974')
ax3.set_ylabel('Latency (ms)', color='#937C2C')
ax3.tick_params(axis='y', labelcolor='#937C2C')
ax3.grid(False)
ax.set_title('Model Cost: Params & Inference Latency', fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(short)

# ── Panel 6: Efficiency — F1 per million params ──
ax = axes[1,2]
eff = [f1s[i]/prm[i] if prm[i]>0 else 0 for i in range(len(loaded))]
bars = ax.bar(x, eff, color='#C44E52', alpha=0.85)
for i,v in enumerate(eff):
    ax.text(i, v, f"{v:.3f}", ha='center', va='bottom', fontsize=7.5)
ax.set_title('Efficiency: Macro-F1 per Million Params', fontweight='bold')
ax.set_ylabel('F1 / M-params')
ax.set_xticks(x); ax.set_xticklabels(short)

plt.tight_layout(rect=[0,0,1,0.98])
plt.savefig(OUTPUT_DIR/'ablation_dashboard.png', dpi=160, bbox_inches='tight')
plt.close()

# ── Also keep the simple staircase (some prefer it for the thesis body) ──
fig, ax = plt.subplots(figsize=(11,6))
ax.plot(x, f1s, 'o-', lw=2, color='#2A9D8F', markersize=8)
for i,v in enumerate(f1s): ax.text(i, v+0.003, f"{v:.3f}", ha='center', fontsize=9)
ax.set_xticks(x); ax.set_xticklabels([r['name'] for r in loaded], rotation=35, ha='right', fontsize=8)
ax.set_ylabel('Test Macro F1'); ax.set_title('EdgeNetV4 Ablation — Cumulative Contribution')
ax.grid(alpha=0.3); plt.tight_layout()
plt.savefig(OUTPUT_DIR/'ablation_staircase.png', dpi=160, bbox_inches='tight'); plt.close()

# ── Console output ──
print("="*72)
print(main_df.to_string(index=False))
print("="*72)
print("\nPer-class F1:")
print(pc_df.to_string(index=False))
print(f"\n📊 Saved:")
print(f"   {OUTPUT_DIR/'ablation_dashboard.png'}   (6-panel comparison)")
print(f"   {OUTPUT_DIR/'ablation_staircase.png'}   (simple line)")
print(f"   {OUTPUT_DIR/'ablation_results.csv'}")
print(f"   {OUTPUT_DIR/'ablation_per_class_f1.csv'}")


Found 7/7 completed runs

             Configuration  Macro_F1 Delta_F1  Defect_Acc  Binary_Acc  Product_Acc  Params_M  Latency_ms   FPS
     1. Baseline (B0, MTL)    0.9464                95.27       98.80        99.94      4.10        5.58 179.2
          2. + B2 backbone    0.9373  -0.0091       94.59       98.98        99.94      7.73        9.00 111.1
       3. + CoordAttention    0.9702  +0.0329       97.30       99.04        99.94      7.75       10.80  92.6
     4. + Multi-scale head    0.9525  -0.0178       95.95       99.16        99.82      7.81       10.19  98.2
  5. + Weighted focal loss    0.9352  -0.0172       94.59       99.10        99.88      7.81        9.77 102.4
         6. + Warm restart    0.9348  -0.0005       93.92       98.44        99.82      7.81       12.18  82.1
7. + Training group (Full)    0.9182  -0.0165       93.24       98.44        99.94      7.80        6.87 145.7

Per-class F1:
             Configuration  contamination   cut  deformation  fracture